In [1]:
import re
from datetime import date, datetime
from typing import List, Optional, Tuple

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webdriver import WebDriver, WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

import src.flights.scrapers.extractors as extractors
import src.flights.utils.utils as utils
from src.flights.models.models import Airport, Flight, SingleSearch
from src.flights.scrapers import utils as scraper_utils

In [2]:
timeout = 10
url = "https://www.google.com/travel/flights/search?tfs=CBwQAhoeEgoyMDI0LTEyLTIyagcIARIDTUlBcgcIARIDTEFYQAFIAXABggELCP___________wGYAQI&tfu=EgoIABAAGAAgAigB&gl=IT&curr=EUR"

In [3]:
options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1200, 2000")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get(url)
WebDriverWait(driver, timeout).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Accept all')]"))).click()

In [4]:
flights_sections = WebDriverWait(driver, timeout).until(EC.visibility_of_all_elements_located((By.CLASS_NAME, "Rk10dc")))
elements = []
for section in flights_sections:
    for row in section.find_elements(By.TAG_NAME, "li"):
        # skip "View more flights" elements
        elements.append(row)

len(elements)

16

In [25]:
def get_duration_in_minutes_from_string(s: str) -> int:
    """
    Returns the duration in minutes from a string.

    :param s: Duration string in the format "X hr Y min" or "X min".
    :return: Duration in minutes.

    Examples:
        3 hr 20 min --> 60*3 + 20 = 200
        20 min --> 20
        5 hr 55 min --> 60*5 + 55 = 355
    """
    assert bool(re.search("hr|min", str(s))), "Invalid duration string format"

    match = re.match(r"(?:(\d+) hr)? ?(?:(\d+) min)?", s)
    h, m = match.groups(default="0")
    return int(h) * 60 + int(m)


def extract_layover_information(element: WebElement) -> Tuple[int, Optional[List[str]], Optional[int]]:
    """
    Extracts layover information from a flight element.

    :param element: <li> WebElement containing flight data.
    :return:
        - Number of stops (int)
        - Layover location(s) Optional(List[str])
        - Layover time (in minutes) Optional(int)
    """
    n_stops, layover_location, layover_time = None, None, None

    layover_section = element.find_element(By.CLASS_NAME, "BbR8Ec")
    layover_text_raw = layover_section.text
    layover_text_list = layover_text_raw.split("\n")
    print(layover_text_list)

    # 1. number of stops
    layover_stops_text = layover_text_list[0]
    if layover_stops_text == "Nonstop":
        return 0, None, None
    else:
        n_stops = int(layover_stops_text.split(" ")[0])

    # 2. layover location
    layover_time_and_location_text = layover_text_list[1]
    location_match = re.search(r"([A-Z]{3}(?:, [A-Z]{3})*)$", layover_time_and_location_text)
    layover_location = location_match.group().split(", ") if location_match else None

    # 3. layover time
    if n_stops >= 2:  # if there are 2 or more stops, layover time is not shown
        return n_stops, layover_location, None

    if layover_location:
        layover_location_str = ", ".join(layover_location)
        layover_time_and_location_text = layover_time_and_location_text.replace(layover_location_str, "").strip()

    layover_time = get_duration_in_minutes_from_string(layover_time_and_location_text)

    return n_stops, layover_location, layover_time

In [27]:
x = extract_layover_information(elements[7])
x

['2 stops', 'ATL, DAL']


(2, ['ATL', 'DAL'], None)